In [ ]:
import re

In [ ]:
import json

In [ ]:
import pickle

In [ ]:
import tqdm

In [ ]:
from dotenv import load_dotenv

In [ ]:
load_dotenv()

False

In [ ]:
import pandas as pd

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:
with open("preds_wo_features.pkl", "rb") as f:
    preds_wo_features = pickle.load(f)

In [ ]:
with open("preds_w_features.pkl", "rb") as f:
    preds_w_features = pickle.load(f)

In [ ]:
with open("titles_with_tags_dict.pkl", "rb") as f:
    titles_with_tags_dict = pickle.load(f)

In [ ]:
with open("artificial_profiles.json", "rb") as f:
    artificial_profiles = json.load(f)

In [ ]:
SYSTEM_PROMPT = """
You are a helpful assistant, which can evaluate, wherether or not a proposed project fits student interests.
Return strictly JSON. Take into account bio and interests of a student.
Example Input:
```
You are machine_learning_researcher student.
Your bio is 'A student, which enjoys solving machine learning problems'.
Your interests are ['machine_learning', deep_learning'].
You are proposed with a project named 'Рекомендательная система студенческих проектов' with corresponding tags ['machine_learning', 'deep_learning', 'recommender_systems'].
```
Exapmle Output:
```json
{'is_valid_recommendation': 1}
```
"""

In [ ]:
def process(df):
    messages = []

    for row in df.iterrows():
        user_name = row[1].user_name
        user_bio = artificial_profiles[user_name]["bio"]
        user_tags = artificial_profiles[user_name]["tags"]

        item_title = row[1].item_title
        item_tags = titles_with_tags_dict[item_title]

        USER_MESSAGE = f"""
        You are {user_name} student.
        Your bio is '{user_bio}'.
        Your interests are {user_tags}.
        You are proposed with a project named '{item_title}' with corresponding tags {item_tags}.
        """

        messages.append({
            "user_name": user_name,
            "item_title": item_title,
            "message":
            (
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_MESSAGE},
            ),
            })

    data = []

    for row in tqdm.tqdm(messages):
        text = processor.apply_chat_template(
            row["message"], tokenize=False, add_generation_prompt=True, enable_thinking=False
        )

        inputs = processor(text=text, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[-1]

        outputs = model.generate(**inputs, max_new_tokens=1024)
        response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

        try:
            json_match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
            if json_match:
                json_str = json_match.group(1)
            else:
                json_str = response
            answer = json.loads(json_str)
        except Exception:
            answer = {"is_valid_recommendation": None}

        try:
            is_valid = answer.get("is_valid_recommendation", None)
            is_valid = int(is_valid)
            data.append((row["user_name"], row["item_title"], is_valid))
        except (TypeError, ValueError):
            pass

    return pd.DataFrame(data=data, columns=["user_name", "item_title", "is_valid"])

In [ ]:
preds_wo_features_scores = process(preds_wo_features)

100%|██████████| 310/310 [03:59<00:00,  1.29it/s]


In [ ]:
preds_w_features_scores = process(preds_w_features)

100%|██████████| 310/310 [03:58<00:00,  1.30it/s]


In [ ]:
preds_wo_features_scores.is_valid.mean()

np.float64(0.8193548387096774)

In [ ]:
preds_w_features_scores.is_valid.mean()

np.float64(0.9032258064516129)

In [ ]:
pwofg = preds_wo_features_scores.groupby(by=["user_name"]).agg({"is_valid": "mean"}).reset_index(drop=False)
pwofg.columns = ["user_name", "mean_score"]
pwofg = pwofg.sort_values(by=["mean_score"], ascending=False).reset_index(drop=True)

In [ ]:
pwofg

,user_name,mean_score
0,digital_marketing_and_media_strategy,1.0
1,data_driven_policy_analyst,1.0
2,cultural_humanities_researcher_media_studies,1.0
3,geopolitics_and_international_relations_expert,1.0
4,global_economics_and_geopolitics_analyst,1.0
5,historical_culture_researcher_media_specialist,1.0
6,international_relations_and_geopolitics_expert,1.0
7,political_science_international_relations_analyst,1.0
8,regional_studies_and_geopolitics_expert,1.0
9,strategic_management_consultant_and_leader,1.0


In [ ]:
pwfg = preds_w_features_scores.groupby(by=["user_name"]).agg({"is_valid": "mean"}).reset_index(drop=False)
pwfg.columns = ["user_name", "mean_score"]
pwfg = pwfg.sort_values(by=["mean_score"], ascending=False).reset_index(drop=True)

In [ ]:
pwfg

,user_name,mean_score
0,cultural_and_media_anthropologist,1.0
1,data_driven_policy_analyst,1.0
2,cultural_humanities_researcher_media_studies,1.0
3,digital_marketing_and_media_strategy,1.0
4,global_economics_and_geopolitics_analyst,1.0
5,historical_culture_researcher_media_specialist,1.0
6,international_relations_and_geopolitics_expert,1.0
7,geopolitics_and_international_relations_expert,1.0
8,political_science_international_relations_analyst,1.0
9,media_journalism_and_cultural_studies_expert,1.0


In [ ]:
with open("preds_wo_features_scores.pkl", "wb") as f:
  pickle.dump(preds_wo_features_scores, f)

In [ ]:
with open("preds_w_features_scores.pkl", "wb") as f:
  pickle.dump(preds_w_features_scores, f)